# Grid search — Muon hybrid & SOAP hybrid

Deux sweeps indépendants, dans le même notebook, avec la même méthodologie 2-phases que `grid_search.ipynb` :

1. **Phase 1** : 2 seeds × grille complète (détection)
2. **Phase 2** : 5 seeds × winner (confirmation, resume de Phase 1)

Les deux optimiseurs sont utilisés en **hybride** : l'optimiseur principal (Muon ou SOAP) sur les **matrices internes 2D**, AdamW catch-all sur le reste (embeddings + biais).

In [ ]:
import os
import numpy as np
import torch as t

import importlib
import pipeline
import plots
importlib.reload(pipeline)
importlib.reload(plots)

from model import Config
from pipeline import (
    OptimizerSpec,
    Trainer,
    grid_search,
    aggregate_grid,
    print_grid_table,
)
from plots import plot_seeds_overlay

t.manual_seed(0)
np.random.seed(0)

In [ ]:
# Config Nanda baseline — partagée par les 2 sweeps.
config = Config(
    p=113,
    d_model=128,
    d_mlp=512,
    num_heads=4,
    n_ctx=3,
    act_type='ReLU',
    frac_train=0.3,
    num_epochs=25_000,    # overridé via grid_search(num_epochs=...)
    seed=0,
)

# Filtre commun: Muon/SOAP sur les matrices internes 2D, AdamW catch-all sinon.
# - 'embed' couvre embed.W_E, pos_embed.W_pos ET unembed.W_U (substring match)
# - ndim >= 2 exclut les biais b_in / b_out (1D)
FILTER_INTERNAL_2D = lambda n, p: p.ndim >= 2 and 'embed' not in n

# Phase 1: détection rapide. Phase 2: confirmation. Même num_epochs -> resume gratuit.
SEEDS_PHASE1 = [0, 1]
SEEDS_PHASE2 = [0, 1, 2, 3, 4]
NUM_EPOCHS   = 25_000

print(f"Device : {config.device}")
print(f"Phase 1 seeds : {SEEDS_PHASE1}")
print(f"Phase 2 seeds : {SEEDS_PHASE2}")

## I. Muon hybride

**Spec** : Muon sur W_K/Q/V/O + W_in/W_out, AdamW catch-all sur W_E + W_pos + W_U + biais.

**HPs variables** : `lr_muon`, `wd_muon`. **Fixes** : `momentum=0.95` (Keller default), AdamW side = baseline Nanda.

In [ ]:
def make_muon_hybrid(lr_muon, wd_muon):
    """Muon sur les matrices internes 2D + AdamW catch-all sur le reste."""
    return [
        OptimizerSpec(
            name='muon',
            lr=lr_muon,
            weight_decay=wd_muon,
            param_filter=FILTER_INTERNAL_2D,
            extra={'momentum': 0.95},
        ),
        OptimizerSpec(
            name='adamw',
            lr=1e-3,
            weight_decay=1.0,
            extra={'betas': (0.9, 0.98)},
        ),
    ]

# Sanity check : visualiser la partition des params
from pipeline import build_optimizers, build_model
model_demo = build_model(config)
_ = build_optimizers(model_demo, make_muon_hybrid(5e-3, 0.5), verbose=True)
del model_demo

In [ ]:
# Grille Muon. Note : Keller default lr=0.02 est trop élevé pour modular addition.
PARAM_GRID_MUON = {
    'lr_muon': [1e-3, 3e-3, 1e-2],   # log-spaced, plus bas que Keller default
    'wd_muon': [0.3, 1.0, 3.0],      # range Nanda-style sur WD
}
SAVE_ROOT_MUON = 'runs/grid/muon_hybrid'

n_combos = len(PARAM_GRID_MUON['lr_muon']) * len(PARAM_GRID_MUON['wd_muon'])
print(f"Muon — Phase 1 : {n_combos} combos × {len(SEEDS_PHASE1)} seeds = {n_combos * len(SEEDS_PHASE1)} runs")
print(f"Muon — save root : {SAVE_ROOT_MUON}/")

In [ ]:
# Phase 1 Muon — fan-out 2 seeds
results_muon_p1 = grid_search(
    config,
    make_muon_hybrid,
    PARAM_GRID_MUON,
    seeds=SEEDS_PHASE1,
    save_root=SAVE_ROOT_MUON,
    num_epochs=NUM_EPOCHS,
    eval_every=50,
    fourier_every=None,
    warmup_steps=10,
    verbose_every=5_000,
    verbose_build=False,
)

In [ ]:
rows_muon_p1 = aggregate_grid(
    results_muon_p1,
    param_keys=list(PARAM_GRID_MUON.keys()),
    acc_thresh=0.99,
    robustness_thresh=0.5,
)
print("=== Muon Phase 1 ===")
print_grid_table(rows_muon_p1, param_keys=list(PARAM_GRID_MUON.keys()))

In [ ]:
# Phase 2 Muon — confirmation 5 seeds sur le winner (resume gratuit)
best_muon = rows_muon_p1[0]
BEST_LR_MUON = best_muon['lr_muon']
BEST_WD_MUON = best_muon['wd_muon']

BEST_GRID_MUON = {'lr_muon': [BEST_LR_MUON], 'wd_muon': [BEST_WD_MUON]}

results_muon_p2 = grid_search(
    config,
    make_muon_hybrid,
    BEST_GRID_MUON,
    seeds=SEEDS_PHASE2,
    save_root=SAVE_ROOT_MUON,      # même root -> reload seeds 0+1
    num_epochs=NUM_EPOCHS,
    eval_every=50,
    warmup_steps=10,
    verbose_every=5_000,
)

rows_muon_p2 = aggregate_grid(results_muon_p2, param_keys=['lr_muon', 'wd_muon'])
print("=== Muon Phase 2 (winner 5 seeds) ===")
print_grid_table(rows_muon_p2, param_keys=['lr_muon', 'wd_muon'])

plot_seeds_overlay(
    results_muon_p2[(BEST_LR_MUON, BEST_WD_MUON)],
    title=f"Muon hybrid — best HP : lr={BEST_LR_MUON}, wd={BEST_WD_MUON} (5 seeds)",
    aggregate='median',
)

## II. SOAP hybride

**Spec** : SOAP sur W_K/Q/V/O + W_in/W_out (mêmes matrices que Muon, pour comparabilité), AdamW catch-all sur le reste.

**HPs variables** : `lr_soap`, `wd_soap`. **Fixes** : `betas=(0.95, 0.95)`, `precondition_frequency=10`, AdamW side identique au sweep Muon.

In [ ]:
def make_soap_hybrid(lr_soap, wd_soap):
    """SOAP sur les matrices internes 2D + AdamW catch-all sur le reste."""
    return [
        OptimizerSpec(
            name='soap',
            lr=lr_soap,
            weight_decay=wd_soap,
            param_filter=FILTER_INTERNAL_2D,
            extra={
                'betas': (0.95, 0.95),             # SOAP default
                'precondition_frequency': 10,      # SOAP default
                'shampoo_beta': -1,                # use betas[1] for shampoo update
            },
        ),
        OptimizerSpec(
            name='adamw',
            lr=1e-3,
            weight_decay=1.0,
            extra={'betas': (0.9, 0.98)},
        ),
    ]

# Sanity check
model_demo = build_model(config)
_ = build_optimizers(model_demo, make_soap_hybrid(3e-3, 0.1), verbose=True)
del model_demo

In [ ]:
# Grille SOAP. WD plus bas que pour Muon/AdamW (SOAP default = 0.01).
PARAM_GRID_SOAP = {
    'lr_soap': [1e-3, 3e-3, 1e-2],     # log-spaced autour de SOAP default 3e-3
    'wd_soap': [0.01, 0.1, 1.0],       # range autour de SOAP default 0.01
}
SAVE_ROOT_SOAP = 'runs/grid/soap_hybrid'

n_combos_soap = len(PARAM_GRID_SOAP['lr_soap']) * len(PARAM_GRID_SOAP['wd_soap'])
print(f"SOAP — Phase 1 : {n_combos_soap} combos × {len(SEEDS_PHASE1)} seeds = {n_combos_soap * len(SEEDS_PHASE1)} runs")
print(f"SOAP — save root : {SAVE_ROOT_SOAP}/")

In [ ]:
# Phase 1 SOAP
results_soap_p1 = grid_search(
    config,
    make_soap_hybrid,
    PARAM_GRID_SOAP,
    seeds=SEEDS_PHASE1,
    save_root=SAVE_ROOT_SOAP,
    num_epochs=NUM_EPOCHS,
    eval_every=50,
    fourier_every=None,
    warmup_steps=10,
    verbose_every=5_000,
    verbose_build=False,
)

In [ ]:
rows_soap_p1 = aggregate_grid(
    results_soap_p1,
    param_keys=list(PARAM_GRID_SOAP.keys()),
    acc_thresh=0.99,
    robustness_thresh=0.5,
)
print("=== SOAP Phase 1 ===")
print_grid_table(rows_soap_p1, param_keys=list(PARAM_GRID_SOAP.keys()))

In [ ]:
# Phase 2 SOAP — confirmation 5 seeds sur le winner
best_soap = rows_soap_p1[0]
BEST_LR_SOAP = best_soap['lr_soap']
BEST_WD_SOAP = best_soap['wd_soap']

BEST_GRID_SOAP = {'lr_soap': [BEST_LR_SOAP], 'wd_soap': [BEST_WD_SOAP]}

results_soap_p2 = grid_search(
    config,
    make_soap_hybrid,
    BEST_GRID_SOAP,
    seeds=SEEDS_PHASE2,
    save_root=SAVE_ROOT_SOAP,
    num_epochs=NUM_EPOCHS,
    eval_every=50,
    warmup_steps=10,
    verbose_every=5_000,
)

rows_soap_p2 = aggregate_grid(results_soap_p2, param_keys=['lr_soap', 'wd_soap'])
print("=== SOAP Phase 2 (winner 5 seeds) ===")
print_grid_table(rows_soap_p2, param_keys=['lr_soap', 'wd_soap'])

plot_seeds_overlay(
    results_soap_p2[(BEST_LR_SOAP, BEST_WD_SOAP)],
    title=f"SOAP hybrid — best HP : lr={BEST_LR_SOAP}, wd={BEST_WD_SOAP} (5 seeds)",
    aggregate='median',
)